In [1]:
import pyspark
from pyspark.sql import SparkSession

In [2]:
# Create SparkSession
spark =  SparkSession.builder \
                    .master("spark://spark-master:7077") \
                    .appName("example") \
                    .config("spark.executor.memory", "2g") \
                    .getOrCreate()
# spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/25 06:40:49 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/02/25 06:40:49 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


#### scenario 7

In [ ]:
# Executor memory = 2 GB
# ~300 MB is reserved for system overhead (not usable by Spark)
# Usable memory ≈ 1.7 GB
# spark.memory.fraction (default 0.6) → 0.6 * 1.7 GB ≈ 1.02 GB
# This ~1 GB is used for execution + storage (unified memory)
# Memory overhead (spark.executor.memoryOverhead) is added on top of executor memory
# JVM heap memory and off-heap memory are also considered separately means off heap memory also add on top of it
# But Spark UI will still only show ~1 GB as Spark memory execution + storage (unified memory).

# Create SparkSession
spark =  SparkSession.builder \
                    .master("spark://spark-master:7077") \
                    .appName("example") \
                    .config("spark.executor.memory", "2g") \
                    .config("spark.executor.memoryOverhead", "300m") \
                    .getOrCreate()
# spark

#### scenario 3

In [ ]:
paquet_df=spark.read.parquet("/app/data/ecommerce.parquet")

26/02/25 06:40:30 WARN TaskSchedulerImpl: Initial job has not accepted any resources; check your cluster UI to ensure that workers are registered and have sufficient resources


In [13]:
paquet_df.rdd.getNumPartitions()

9

In [ ]:
spark.createOrReplaceTempView(paquet_df, "ttable")

In [8]:
spark.sql("SELECT order_id, user_id, product_id, category, price FROM (SELECT * FROM ttable WHERE order_status='returned') WHERE price>100").explain(True)

== Parsed Logical Plan ==
'Project ['order_id, 'user_id, 'product_id, 'category, 'price]
+- 'Filter ('price > 100)
   +- 'SubqueryAlias __auto_generated_subquery_name
      +- 'Project [*]
         +- 'Filter ('order_status = returned)
            +- 'UnresolvedRelation [ttable], [], false

== Analyzed Logical Plan ==
order_id: bigint, user_id: bigint, product_id: bigint, category: string, price: double
Project [order_id#0L, user_id#1L, product_id#2L, category#3, price#4]
+- Filter (price#4 > cast(100 as double))
   +- SubqueryAlias __auto_generated_subquery_name
      +- Project [order_id#0L, user_id#1L, product_id#2L, category#3, price#4, quantity#5L, payment_method#6, order_status#7, order_ts#8, country#9]
         +- Filter (order_status#7 = returned)
            +- SubqueryAlias ttable
               +- View (`ttable`, [order_id#0L, user_id#1L, product_id#2L, category#3, price#4, quantity#5L, payment_method#6, order_status#7, order_ts#8, country#9])
                  +- Relation [

In [19]:
spark.sql("SELECT order_id, user_id, product_id, category, price FROM (SELECT * FROM ttable WHERE order_status='returned') WHERE price>100").explain()

== Physical Plan ==
*(1) Project [order_id#0L, user_id#1L, product_id#2L, category#3, price#4]
+- *(1) Filter (((isnotnull(order_status#7) AND isnotnull(price#4)) AND (order_status#7 = returned)) AND (price#4 > 100.0))
   +- *(1) ColumnarToRow
      +- FileScan parquet [order_id#0L,user_id#1L,product_id#2L,category#3,price#4,order_status#7] Batched: true, DataFilters: [isnotnull(order_status#7), isnotnull(price#4), (order_status#7 = returned), (price#4 > 100.0)], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/app/data/ecommerce.parquet], PartitionFilters: [], PushedFilters: [IsNotNull(order_status), IsNotNull(price), EqualTo(order_status,returned), GreaterThan(price,100..., ReadSchema: struct<order_id:bigint,user_id:bigint,product_id:bigint,category:string,price:double,order_status...




#### scenario 4

In [1]:
paquet_df=spark.read.parquet("/app/data/ecommerce.parquet")
spark.createOrReplaceTempView(paquet_df, "ttable")

In [ ]:
# After building the optimized physical plan -> In whole stage codegen Spark fuses multiple operators into a single JVM function per partition.

df=spark.sql("SELECT order_id, user_id, product_id, category, price FROM (SELECT * FROM ttable WHERE order_status='returned') WHERE price>100")
df.explain("codegen")

#### scenario 5

RDDs are just wrappers for partition-level computation
An RDD represents a distributed dataset, split into partitions.
Each partition has a compute function, which Spark calls to produce rows.
That compute function executes whatever logic you give it — in your case, the generated Java code from whole-stage codegen.
So, the RDD itself doesn’t store filters, projections, or joins abstractly. It just knows:
“For this partition, run this function to produce rows.”
Roughly speaking: “RDD = partition-level execution wrapper”

#### scenario 6

In [ ]:
df=spark.sql("SELECT order_id, user_id, product_id, category, price FROM (SELECT * FROM ttable WHERE order_status='returned') WHERE price>100")

DataFrame / SQL   ← explain() works here
      ↓
Catalyst Optimizer
      ↓
Physical Plan
      ↓
RDD Execution Layer ← toDebugString() works here

In [18]:
rdd = df.rdd
print(rdd.toDebugString().decode("utf-8"))

(9) MapPartitionsRDD[28] at javaToPython at <unknown>:0 []
 |  MapPartitionsRDD[27] at javaToPython at <unknown>:0 []
 |  SQLExecutionRDD[26] at javaToPython at <unknown>:0 []
 |  MapPartitionsRDD[25] at javaToPython at <unknown>:0 []
 |  MapPartitionsRDD[24] at javaToPython at <unknown>:0 []
 |  FileScanRDD[23] at javaToPython at <unknown>:0 []


In [19]:
df.explain("formatted")

== Physical Plan ==
* Project (4)
+- * Filter (3)
   +- * ColumnarToRow (2)
      +- Scan parquet  (1)


(1) Scan parquet 
Output [6]: [order_id#14L, user_id#15L, product_id#16L, category#17, price#18, order_status#21]
Batched: true
Location: InMemoryFileIndex [file:/app/data/ecommerce.parquet]
PushedFilters: [IsNotNull(order_status), IsNotNull(price), EqualTo(order_status,returned), GreaterThan(price,100.0)]
ReadSchema: struct<order_id:bigint,user_id:bigint,product_id:bigint,category:string,price:double,order_status:string>

(2) ColumnarToRow [codegen id : 1]
Input [6]: [order_id#14L, user_id#15L, product_id#16L, category#17, price#18, order_status#21]

(3) Filter [codegen id : 1]
Input [6]: [order_id#14L, user_id#15L, product_id#16L, category#17, price#18, order_status#21]
Condition : (((isnotnull(order_status#21) AND isnotnull(price#18)) AND (order_status#21 = returned)) AND (price#18 > 100.0))

(4) Project [codegen id : 1]
Output [5]: [order_id#14L, user_id#15L, product_id#16L, cat

#### scenario 2

In [ ]:
df = spark.createDataFrame([Row(name="Alice", age=2), Row(name="Bob", age=5)])

In [7]:
df.rdd.getNumPartitions() # why 8 partitions because i have not set paralization level while crating DF so saprk assighn 'spark.default.parallelism'->assighn partitions base on cores in cluster

8

In [8]:
df.rdd.glom().collect()

[[], [], [], [Row(name='Alice', age=2)], [], [], [], [Row(name='Bob', age=5)]]

In [10]:
df = df.coalesce(2)

In [11]:
df.rdd.glom().collect() # glom: Converts each partition into a Python list

[[Row(name='Alice', age=2)], [Row(name='Bob', age=5)]]

In [12]:
df = df.coalesce(1)

In [13]:
df.rdd.glom().collect()

[[Row(name='Alice', age=2), Row(name='Bob', age=5)]]

In [14]:
df = df.coalesce(0) # error IllegalArgumentException: requirement failed: Number of partitions (0) must be positive.

#### scenario 1

In [3]:
paquet_df=spark.read.parquet("/app/data/ecommerce.parquet")
df.rdd.getNumPartitions() # why 9 when load data from file it make splits where max partition size 128MB reached it made 9 partitions it means file size  is 9*128 MBs ~ 1152 MBs

In [15]:
df=paquet_df.filter((paquet_df['price']>100) & (paquet_df['price']<110))
df.show()

+--------+-------+----------+-----------+------+--------+--------------+------------+-------------------+-------+
|order_id|user_id|product_id|   category| price|quantity|payment_method|order_status|           order_ts|country|
+--------+-------+----------+-----------+------+--------+--------------+------------+-------------------+-------+
|      27| 411265|     24566|    fashion|100.53|       4|   credit_card|    returned|2022-02-09 12:06:44|     FR|
|      34| 919262|     19305|      books|105.88|       4|   credit_card|    returned|2023-10-02 08:30:43|     UK|
|      63| 466903|     33245|    fashion|102.73|       3|           upi|   delivered|2021-09-01 12:18:01|     FR|
|     130| 675308|     32241|       home|105.47|       4|   credit_card|      placed|2021-06-22 22:37:42|     DE|
|     179| 634594|     23543|electronics|102.37|       1|   credit_card|     shipped|2021-01-13 13:11:49|     FR|
|     229| 388244|     74597|      books|105.64|       2|   credit_card|   delivered|202

In [20]:
records=df.rdd.glom().collect()
print(len(records))
print("Number of Records in first partition:",len(records[0]))
print("Number of Records in second partition:",len(records[2]))
print("Number of Records in third partition:",len(records[3]))
print("First 4 records in first patition:",records[0][0:4])
print("First recrd of first patition:",records[0][0])

9
Number of Records in first partition: 121260
Number of Records in second partition: 100885
Number of Records in third partition: 120634
First 4 records in first patition: [Row(order_id=27, user_id=411265, product_id=24566, category='fashion', price=100.53, quantity=4, payment_method='credit_card', order_status='returned', order_ts=datetime.datetime(2022, 2, 9, 12, 6, 44), country='FR'), Row(order_id=34, user_id=919262, product_id=19305, category='books', price=105.88, quantity=4, payment_method='credit_card', order_status='returned', order_ts=datetime.datetime(2023, 10, 2, 8, 30, 43), country='UK'), Row(order_id=63, user_id=466903, product_id=33245, category='fashion', price=102.73, quantity=3, payment_method='upi', order_status='delivered', order_ts=datetime.datetime(2021, 9, 1, 12, 18, 1), country='FR'), Row(order_id=130, user_id=675308, product_id=32241, category='home', price=105.47, quantity=4, payment_method='credit_card', order_status='placed', order_ts=datetime.datetime(2021,

#### scenario 

In [ ]:
from pyspark.sql import Row
df = spark.createDataFrame([Row(name="Alice", age=2), Row(name="Bob", age=5)])

In [18]:
# only narrow transformation
df.show()

In [ ]:
# wide transformation
df.count()

#### scenario ### Auto BroadCast Join

In [21]:
large_df = spark.range(0, 5_000_000) \
    .withColumn("key", (col("id") % 100)) \
    .withColumn("value", rand())

small_df = spark.range(0, 100) \
    .withColumnRenamed("id", "key") \
    .withColumn("category", (col("key") * 10))

normal_join = large_df.join(small_df, on="key")

normal_join.explain(True)

In [ ]:
normal_join.count()

In [ ]:
normal_join.show()